In [ ]:
from IPython.display import display
import time
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from IPython.display import HTML
import numpy as np
import pandas as pd
import torch
from genkit import GaussianFlowLinear
from genkit._noise import sample_scaled_isotropic_alpha_stable
from genkit.nn import MLPModel
from genkit.training import train
from genkit.metrics import mmd_rbf, sliced_wasserstein, tail_coverage_error

plt.rcParams["animation.embed_limit"] = 100


In [ ]:
class GuidedGaussianFlowLinear:
    def __init__(self, flow: GaussianFlowLinear, tail_radius: float = 2.0, tail_scale: float = 0.5, lambda_schedule=None, guidance_scale: float = 1.0) -> None:
        self.flow = flow.flow if isinstance(flow, GuidedGaussianFlowLinear) else flow

        if not isinstance(self.flow, GaussianFlowLinear):
            raise TypeError("flow must be a trained GaussianFlowLinear or GuidedGaussianFlowLinear.")

        self.tail_radius = float(tail_radius)
        self.tail_scale = float(tail_scale)
        if self.tail_scale <= 0.0:
            raise ValueError("tail_scale must be positive.")
        self.lambda_schedule = lambda_schedule
        self.guidance_scale = float(guidance_scale)

    def __getattr__(self, name):
        if "flow" not in self.__dict__:
            raise AttributeError(name)
        return getattr(self.flow, name)

    def drift(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            velocity = self.flow._net(x, t)

        if self.guidance_scale == 0.0:
            return velocity

        with torch.enable_grad():
            x_req = x.detach().requires_grad_(True)
            tau = torch.nn.functional.softplus((x_req.flatten(1).norm(dim=1) - self.tail_radius) / self.tail_scale)
            grad = torch.autograd.grad(tau.sum(), x_req)[0]

        lbda = 1.0 if self.lambda_schedule is None else self.lambda_schedule(t)
        lbda = torch.as_tensor(lbda, device=x.device, dtype=x.dtype)
        while lbda.ndim < x.ndim:
            lbda = lbda.unsqueeze(-1)

        return velocity + self.guidance_scale * lbda * grad

    @torch.no_grad()
    def _sample(self, n_samples: int | None = None, return_trajectory: bool = True, sample_source: torch.Tensor | None = None):
        self.flow._net.eval()

        _, x = self.flow._resolve_sample_source(n_samples, sample_source)
        timesteps = torch.linspace(self.flow._t_min, self.flow._t_max, self.flow._sample_steps + 1, device=self.flow._device, dtype=self.flow._fdtype)
        trajectory = [x.detach()] if return_trajectory else None

        for t0, t1 in zip(timesteps[:-1], timesteps[1:]):

            t = t0.reshape(1, 1).expand(x.size(0), 1)
            x = x + (t1 - t0) * self.drift(x, t)

            if trajectory is not None:
                trajectory.append(x.detach())

        return x, trajectory

    @torch.no_grad()
    def sample(self, n_samples: int | None = None, sample_source: torch.Tensor | None = None):
        x, _ = self._sample(n_samples=n_samples, return_trajectory=False, sample_source=sample_source)
        return x


In [ ]:
seed = 0
torch.manual_seed(seed)
device = 'cpu'
dtype = torch.float32
batch_size = 512
lr = 1e-3
n_train = 10_000
n_epochs = 256
n_steps = 128
lim = 4.0
n_vis = 150
n_trails = 150
n_grid = 15
alpha = 1.7
dim = 2
train_data = sample_scaled_isotropic_alpha_stable(n_samples=n_train, dim=dim, alpha=alpha, device=device, dtype=dtype)
guided_kwargs = [("GuidedGFL05", {"tail_radius": 2.0, "tail_scale": 0.5, "guidance_scale": 0.5}),
                 ("GuidedGFL10", {"tail_radius": 2.0, "tail_scale": 0.5, "guidance_scale": 1.0}),
                 ]


In [ ]:
models, diagnostics = {}, {}

model = GaussianFlowLinear(MLPModel(dim=dim, width=256, depth=3), dim=dim, n_steps=n_steps, t_min=0.0, t_max=1.0, sampler="euler", sample_steps=n_steps, device=device, fdtype=dtype)
models["GFL"], diagnostics["GFL"] = train(model, train_data, batch_size=batch_size, n_epochs=n_epochs, lr=lr, device=device, lr_schedule="constant")

for name, kwargs in guided_kwargs:
    models[name] = GuidedGaussianFlowLinear(models["GFL"], **kwargs)


In [ ]:
n_tail = 100_000
n_pairwise = 1_000
quantiles = [0.9, 0.99, 0.999]
probs = torch.tensor([1.0 - q for q in quantiles], device=device, dtype=dtype)

x_ref = sample_scaled_isotropic_alpha_stable(n_samples=n_tail, dim=dim, alpha=alpha, device=device, dtype=dtype)
x_source = next(iter(models.values()))._sample_source(n_tail)
metric_rows = []

for name, model in models.items():

    start_time = time.time()
    print(f"Evaluating '{name}'...", end="")

    x_gen = model.sample(sample_source=x_source)

    row = {"model": name,
           "mmd_rbf": mmd_rbf(x_ref[:n_pairwise], x_gen[:n_pairwise]),
           "sliced_w2": sliced_wasserstein(x_ref[:n_pairwise], x_gen[:n_pairwise]),
           }

    tce = tail_coverage_error(x_ref, x_gen, probs=probs, tail="upper", mode="log", reduction="none")
    for q, err in zip(quantiles, tce.mean(dim=1).cpu().numpy()):
        row[f"tce_{q}"] = float(err)
    metric_rows.append(row)

    print(f"done in {time.time() - start_time:.1f} s", flush=True)

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)


In [ ]:
target_preview = sample_scaled_isotropic_alpha_stable(n_samples=3_000, dim=dim, alpha=alpha, device=device, dtype=dtype)
sample_source = next(iter(models.values()))._sample_source(n_vis)
visuals = {}

for name, model in models.items():

    grid_axis = torch.linspace(-lim, lim, n_grid, device=model._device, dtype=model._fdtype)
    field_points = torch.cartesian_prod(grid_axis, grid_axis)

    _, trajectory_steps = model._sample(sample_source=sample_source)

    trajectory = torch.stack(trajectory_steps).detach()
    flow_times = torch.linspace(model._t_min, model._t_max, len(trajectory_steps), device=model._device, dtype=model._fdtype)
    field_times = [t.expand(field_points.shape[0], 1) for t in flow_times]

    if isinstance(model, GuidedGaussianFlowLinear):
        field_vectors = torch.stack([model.drift(field_points, t) for t in field_times]).detach()
        with torch.no_grad():
            base_vectors = torch.stack([model.flow._net(field_points, t) for t in field_times]).detach()
    else:
        with torch.no_grad():
            field_vectors = torch.stack([model._net(field_points, t) for t in field_times]).detach()
        base_vectors = field_vectors

    correction_vectors = field_vectors - base_vectors

    visuals[name] = {
        "trajectory": trajectory.cpu().numpy(),
        "field_points": field_points.cpu().numpy(),
        "field_vectors": field_vectors.cpu().numpy(),
        "flow_times": flow_times.cpu().numpy(),
        "correction_vectors": correction_vectors.cpu().numpy(),
        "correction_norm": float(correction_vectors.norm(dim=-1).mean().item()),
    }

target_np = target_preview.cpu().numpy()


In [ ]:
n_frames = min(data["trajectory"].shape[0] for data in visuals.values())
model_items = list(visuals.items())
row_labels = ["trajectory", "vector field", "correction"]

fig, axes = plt.subplots(len(row_labels), len(model_items), figsize=(4 * len(model_items), 4 * len(row_labels)), sharex=True, sharey=True, squeeze=False)
panels = []

for col, (name, data) in enumerate(model_items):

    trajectory_np = data["trajectory"]
    field_points_np = data["field_points"]
    field_vectors_np = data["field_vectors"]
    correction_vectors_np = data["correction_vectors"]
    flow_times_np = data["flow_times"]
    correction_norm = data["correction_norm"]
    trail_idx = np.unique(np.linspace(0, trajectory_np.shape[1] - 1, min(n_trails, trajectory_np.shape[1]), dtype=int))

    trajectory_ax, field_ax, correction_ax = axes[:, col]

    trajectory_ax.scatter(target_np[:, 0], target_np[:, 1], s=7, color="0.6", alpha=0.2, linewidths=0)
    trail_lines = LineCollection([], colors="tab:blue", linewidths=2.0, alpha=0.1)
    trajectory_ax.add_collection(trail_lines)
    sample_cloud = trajectory_ax.scatter(trajectory_np[0, :, 0], trajectory_np[0, :, 1], s=11, color="tab:blue", alpha=0.4, linewidths=0)

    field_arrows = field_ax.quiver(field_points_np[:, 0], field_points_np[:, 1], field_vectors_np[0, :, 0], field_vectors_np[0, :, 1],
                                  angles="xy", scale_units="xy", scale=5, width=0.002, color="tab:blue", alpha=0.4)

    correction_arrows = correction_ax.quiver(field_points_np[:, 0], field_points_np[:, 1], correction_vectors_np[0, :, 0], correction_vectors_np[0, :, 1],
                                             angles="xy", scale_units="xy", scale=5, width=0.002, color="tab:red", alpha=0.5)
    correction_ax.text(0.02, 0.98, f"mean |correction| = {correction_norm:.2f}", transform=correction_ax.transAxes, ha="left", va="top")

    time_texts = [ax.text(0.02, 0.90, "", transform=ax.transAxes, ha="left", va="top") for ax in axes[:, col]]
    axes[0, col].set_title(name)

    panels.append((trajectory_np, field_vectors_np, correction_vectors_np, flow_times_np, trail_idx, trail_lines, sample_cloud, field_arrows, correction_arrows, time_texts))

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label)

for ax in axes.ravel():
    ax.set(aspect="equal", xlim=(-2 * lim, 2 * lim), ylim=(-2 * lim, 2 * lim))
    ax.set_xticks([])
    ax.set_yticks([])

fig.tight_layout()


def update(frame):
    i = int(frame)
    artists = []
    for panel in panels:
        trajectory_np, field_vectors_np, correction_vectors_np, flow_times_np = panel[:4]
        trail_idx, trail_lines, sample_cloud, field_arrows, correction_arrows, time_texts = panel[4:]

        trail_lines.set_segments([trajectory_np[: i + 1, j, :] for j in trail_idx])
        sample_cloud.set_offsets(trajectory_np[i])
        field_arrows.set_UVC(field_vectors_np[i, :, 0], field_vectors_np[i, :, 1])
        correction_arrows.set_UVC(correction_vectors_np[i, :, 0], correction_vectors_np[i, :, 1])

        for time_text in time_texts:
            time_text.set_text(f"t = {flow_times_np[i]:.3f}")

        artists.extend([trail_lines, sample_cloud, field_arrows, correction_arrows, *time_texts])
    return artists

anim = FuncAnimation(fig, update, frames=n_frames, interval=80, repeat=True, blit=False)
html = HTML(anim.to_jshtml(default_mode="loop"))

plt.close(fig)


In [ ]:
html